# TASK 1. ENVIRONMENT SETUP & BUILD AN AI AGENT WITH NO MEMORY

In [ ]:
!apt-get update
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,090 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,332 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:14 htt

In [ ]:
!pip install openai-agents==0.10.5
!ollama pull llama3.2


Error: could not connect to ollama server, run 'ollama serve' to start it


In [ ]:
import subprocess
import time
import requests
from IPython.display import display, Markdown

OLLAMA_API_URL = "http://127.0.0.1:11434"
OLLAMA_BASE_URL = f"{OLLAMA_API_URL}/v1"
MODEL_NAME = "llama3.2"


def print_markdown(text: str):
    display(Markdown(text))


def wait_for_ollama(timeout_seconds: int = 30) -> bool:
    """Wait until Ollama's local HTTP API is reachable."""
    deadline = time.time() + timeout_seconds

    while time.time() < deadline:
        try:
            response = requests.get(f"{OLLAMA_API_URL}/api/tags", timeout=2)
            if response.status_code == 200:
                return True
        except requests.RequestException:
            time.sleep(1)

    return False


def start_ollama_server():
    """Start Ollama only if it is not already running."""
    if wait_for_ollama(timeout_seconds=3):
        print("Ollama is already running.")
        return

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    if not wait_for_ollama(timeout_seconds=30):
        raise RuntimeError("Ollama did not start. Check the Colab runtime logs.")

    print("Ollama server started.")


start_ollama_server()


Ollama is already running.


In [ ]:
# Load environment variables and configure client
# from dotenv import load_dotenv
# load_dotenv()
# openai_api_key = os.getenv("OPENAI_API_KEY")

# Pull the model after the server is running
!ollama pull llama3.2

# Verify Ollama can see the model
!ollama list
!curl -s http://127.0.0.1:11434/v1/models



NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    
{"object":"list","data":[{"id":"llama3.2:latest","object":"model","created":1783078353,"owned_by":"library"}]}


In [ ]:
from openai import AsyncOpenAI
from agents import (
    Agent,
    Runner,
    SQLiteSession,
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)

# Disable tracing because we are using the fake Ollama API key, not a real OpenAI API key.
set_tracing_disabled(True)

# Configure Agents SDK to use Ollama's OpenAI-compatible endpoint.
ollama_client = AsyncOpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama",
)

set_default_openai_client(ollama_client)
set_default_openai_api("chat_completions")

print("Agents SDK configured to use Ollama.")


Agents SDK configured to use Ollama.


In [ ]:
finance_advisor_instructions = """
Context:
You are a friendly personal finance advisor who helps users track spending and make better budgeting decisions.

Instructions:
- Every time the user mentions spending, extract:
  - amount
  - currency if given
  - category such as groceries, transport, eating out, rent, entertainment, etc.
- Use conversation history to keep:
  - a running total of all expenses
  - totals per category
- When the user asks for advice or a summary:
  - calculate approximate totals from the conversation so far
  - identify 1–3 categories where they are spending the most
  - suggest specific, practical ways to reduce spending next week.
- Be concise, factual, and non-judgmental.
- Do not invent expenses, categories, budgets, or dates that the user did not mention.

Output:
- When logging expenses, briefly confirm what you recorded.
- When giving advice or summaries, use Markdown with:
  - a short summary sentence
  - a bullet list of key numbers
  - 2–3 concrete suggestions to improve the budget.
"""

finance_advisor_agent = Agent(
    name="Finance Advisor",
    instructions=finance_advisor_instructions,
    model="llama3.2",
)

print("Finance Advisor agent created.")


Finance Advisor agent created.


# TASK 2. RUN AN AI AGENT WITH NO MEMORY

In [ ]:
user_input1 = "I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week."

print_markdown(f"**You:** {user_input1}")

response1 = await Runner.run(
    starting_agent=finance_advisor_agent,
    input=user_input1,
)

print_markdown(f"**🤖 Agent:**\n\n{response1.final_output}")

**You:** I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week.

**🤖 Agent:**

Confirmed expenses:

- **Groceries**: $120 (USD)
- **Transport**: 
  - **Uber**: $40 (USD)

Total spending so far: approximately **$200**

Let's take a look at the categories where you spent the most:
* Groceries
* Transport (specifically, Uber)

Here are some suggestions to improve your budget for next week:

1. Consider meal planning and prep to reduce dining out expenses.
2. Explore alternative transportation options, such as public transport or carpooling, to save on Uber costs.
3. Look for ways to optimize your grocery shopping routine to avoid unnecessary purchases.

Do you have any questions about these suggestions or would you like me to suggest more ideas?

In [ ]:
user_input2 = "My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?"

print_markdown(f"**You:** {user_input2}")

response2 = await Runner.run(
    starting_agent=finance_advisor_agent,
    input=user_input2,
)

print_markdown(f"**🤖 Agent without memory:**\n\n{response2.final_output}")

**You:** My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?

**🤖 Agent without memory:**

**Weekly Budget Summary**
==========================

Based on our previous conversation, here's a summary of your expenses:

* **Category:** No specific categories mentioned
* **Total spent this year:** Not specified (we're just starting!)

Since we haven't logged any expenses yet, I'll assume it's a blank slate. What would you like to talk about or start tracking?

# TASK 3. ADD MEMORY TO THE AGENT USING SQLITESESSION

In [ ]:
finance_session = SQLiteSession("finance_conversation")


async def message_gpt2(prompt: str, session: SQLiteSession):
    """Run the Finance Advisor with memory using the provided SQLiteSession."""
    print_markdown(f"**You:** {prompt}")

    response = await Runner.run(
        starting_agent=finance_advisor_agent,
        input=prompt,
        session=session,
    )

    print_markdown(f"**🤖 Agent:**\n\n{response.final_output}")
    return response.final_output


response1_memory = await message_gpt2(
    "I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week.",
    finance_session,
)


**You:** I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week.

**🤖 Agent:**

Here's what I've recorded:

* **Weekly Expenses:**
	+ Groceries: $120
	+ Uber: $40
	+ Restaurants: $60
	Total: $220
* **Category Totals:**
	+ Groceries: $120
	+ Transportation (Uber): $40
	+ Dining: $60

Please confirm if I've logged your expenses correctly. 

Also, what's next? Want to review your spending or seek advice on how to reduce expenses next week?

In [ ]:
response2_memory = await message_gpt2(
    "My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?",
    finance_session,
)



**You:** My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?

**🤖 Agent:**

**Summary:**
You've overspent by $30 this week.

**Key Numbers:**

* Total spent: $220
* Budgeted amount: $250

Here are some suggestions to help you reduce spending next week:

* **Optimize your Uber usage**: Consider carpooling, using public transport, or canceling your subscription service for next week.
* **Meal planning and cooking**: Plan your meals in advance and cook at home instead of dining out next week. You can allocate $30 for a flexible weekly meal budget.
* **Grocery shopping**: Review your grocery list and avoid impulse buys. Compare prices and consider buying groceries during sales or discounts next week.

By implementing these strategies, you may be able to bring your spending within your budget range next week.

In [ ]:
response3_memory = await message_gpt2(
    "I also spent $30 on coffee. What is my total spending now?",
    finance_session,
)

**PRACTICE OPPORTUNITY:**  
- **Using OpenAI Agents SDK, create a new AI agent named `Wellness Coach` that always suggests one simple weekly health habit.**  
    - **1. Write the agent’s instructions; tell it to give exactly one small, realistic habit per week and avoid repeating previous habits.**  
    - **2. Use the latest `gpt-5-mini` model.**  
    - **3. Save the conversation history with `SQLiteSession` so it remembers the user’s name, lifestyle, and previously suggested habits.**  
    - **4. Ask the agent:**  
      **`My name is Tamara. I work long hours at a desk and feel low energy in the afternoon. Suggest one simple health habit I can try next week.`**  
    - **5. Display the agent’s answer.**  
    - **6. Then ask a follow-up question (same session) asking for a NEW habit that builds on the user’s progress and does not repeat the previous suggestion:**  
      **`Assume I successfully followed your first habit for one week. Suggest a NEW habit for the following week that builds on my progress (do not repeat the previous habit).`**  
    - **Hint: Use `Runner.run()` to send both messages to the agent and print the replies.**


In [ ]:
wellness_coach_instructions = """
You are a supportive health and wellness coach called Wellness Coach.
Your job is to help busy professionals build better habits one week at a time.

Rules:
- Always suggest exactly ONE simple, realistic habit per reply.
- Make the habit small and specific, such as "take a 10-minute walk after lunch on weekdays".
- Give a brief explanation of why this habit helps.
- If the user talks to you again in the same session, remember the habits you already suggested and do not repeat them.
- Suggest something new that builds on their progress.
- Keep the tone encouraging and non-judgmental.
"""

wellness_coach_agent= Agent(name="Wellness Coach", instructions= wellness_coach_instructions, model="llama3.2")
print("Wellness Coach Created!")

user_input1= "My name is Tamara. I work long hours at a desk and feel low energy in the afternoon. Suggest one simple health habit I can try next week."

print_markdown(f"**You:** {user_input1}")

response1= await Runner.run(starting_agent= wellness_coach_agent, input=user_input1)
print_markdown(f"**🤖 Agent:**\n\n{response1.final_output}")

Wellness Coach Created!


**You:** My name is Tamara. I work long hours at a desk and feel low energy in the afternoon. Suggest one simple health habit I can try next week.

**🤖 Agent:**

Hi Tamara! I'm glad you're willing to make some changes.

Considering your job requires long hours of sitting, I'd like to suggest a simple habit to boost your energy levels later in the day. Try this: "Take 5 deep breaths every hour at work."

Yes, just that – five conscious breaths. Sit comfortably, close your eyes (or gaze softly), and focus on inhaling deeply through your nose for about 4 seconds, then exhale slowly through your mouth for another 4 seconds. This habit can help calm your mind, reduce stress, and increase oxygen flow to your brain.

Give it a try next week and see how it goes!

In [ ]:
user_input2 = "Assume I successfully followed your first habit for one week. Suggest a NEW habit for the following week that builds on my progress (do not repeat the previous habit)."

print_markdown(f"**You:** {user_input2}")

response2 = await Runner.run(
    starting_agent=wellness_coach_agent,
    input=user_input2,
)

print_markdown(f"**🤖 Agent without memory:**\n\n{response2.final_output}")

**You:** Assume I successfully followed your first habit for one week. Suggest a NEW habit for the following week that builds on my progress (do not repeat the previous habit).

**🤖 Agent without memory:**

I'm so proud of you for sticking to our habit from last week! It sounds like taking care of yourself was a priority, and that's awesome. For this week, I'd like to suggest a new habit that builds on your progress:

**Habit for this week:** Write down three things you're grateful for each morning before starting your day.

Why it helps: Starting your day with gratitude can set a positive tone and help you focus on what's truly important. Research has shown that practicing gratitude can lower stress levels, improve mood, and even boost immune function. By making gratitude a daily habit, you'll be priming yourself for a more mindful and optimistic approach to the week ahead.

Remember, take just 1-2 minutes each morning to jot down three things that come to mind – it could be something as simple as a good cup of coffee or a beautiful sunset. This habit will help you stay present, appreciate the little things, and cultivate a more positive mindset.

In [ ]:
wellness_session = SQLiteSession("wellness_conversation")

async def message_gpt_wellness(prompt: str, session: SQLiteSession):
  "Run the Wellness Coach with memory using the provided SQLiteSession."
  print_markdown(f"**You:** {prompt}")

  response = await Runner.run(
      starting_agent=wellness_coach_agent,
      input=prompt,
      session=session,
  )

  print_markdown(f"**🤖 Agent:**\n\n{response.final_output}")
  return response.final_output

response1_memory = await message_gpt_wellness(
    "My name is Tamara. I work long hours at a desk and feel low energy in the afternoon. Suggest one simple health habit I can try next week.",
    wellness_session,
)

**You:** My name is Tamara. I work long hours at a desk and feel low energy in the afternoon. Suggest one simple health habit I can try next week.

**🤖 Agent:**

Tamara! I'm so glad you're taking steps to prioritize your well-being.

Considering your long hours at a desk, I'd like to suggest a simple habit that might help boost your energy levels later in the day: **Take a 5-minute stretch break every hour**.

 Whenever you feel alert and awake (usually around lunchtime or mid-afternoon), get up from your desk and do some quick stretches. You can touch your toes, shoulder rolls, neck stretches, or even just some wrist circles. This brief pause will help increase blood flow, reduce tension, and oxygenate your brain.

Remember, it's not about exercising for an extended period; just a few minutes of gentle stretching can make a big difference in how you feel. Plus, getting up from a desk helps prevent prolonged sitting, which is essential to combat that mid-day slump!

How does this sound?

In [ ]:
response2_memory = await message_gpt_wellness(
    "Assume I successfully followed your first habit for one week. Suggest a NEW habit for the following week that builds on my progress (do not repeat the previous habit).",
    wellness_session,
)

**You:** Assume I successfully followed your first habit for one week. Suggest a NEW habit for the following week that builds on my progress (do not repeat the previous habit).

**🤖 Agent:**

Congratulations on completing your first week!

Since you've already established a consistent 10-minute walk after lunch, this week let's focus on incorporating more mindful eating habits. Here's a new suggestion:

**Habit:** Eat one serving of fruits or vegetables with every meal.

This habit helps in several ways: it ensures that you're fueling your body with nutrient-rich foods, which can lead to improved energy levels and a healthier digestive system. Eating mindfully also encourages you to pay attention to your hunger and fullness cues, making it easier to maintain a balanced diet. Plus, a daily dose of fruits or veggies will boost your immune system and support overall well-being.

Start with small steps – begin by focusing on one meal per day (e.g., breakfast) initially, and then expand to two meals, until you reach a point where eating every meal with some fruit or veggie feels natural. Remember, it's all about setting a positive tone for healthy eating from the beginning!

How do you feel about this new habit, and are there any questions or concerns I can help with?

In [ ]:
response3_memory = await message_gpt_wellness(
    "I am stressed out",
    wellness_session,
)

**You:** I am stressed out

**🤖 Agent:**

Tamara, taking care of yourself when you're feeling stressed is super important.

Building on the idea from earlier, I'd like to suggest a habit that combines stress relief with a mindfulness practice: **Write down three things you're grateful for each day before bed**.

Take just 2-3 minutes before drifting off to sleep to jot down three things, no matter how small they may seem. It could be something as simple as a good cup of coffee, a beautiful sunset, or a supportive friend. Focusing on what's good in your life helps shift your perspective and can calm those stressors.

By incorporating gratitude into your daily routine, you'll not only cultivate a more positive mindset but also prepare yourself for a restful night's sleep, which is essential for emotional regulation and overall well-being.

Give it a try, Tamara!